# 01b — FP32 baz model eğitimi (çıktılar doğrudan Google Drive'a)

**Proje:** Uç Cihazlar İçin FPGA Tabanlı BERT-Tiny Hızlandırıcısı

Bu not defteri `01` ile aynı eğitimi yapar, ancak tüm çıktıları **eğitim sırasında doğrudan Drive'a** yazar.
Böylece Colab oturumu kapansa bile en iyi epoch'un modeli kaybolmaz.

> **Çalışma zamanı:** T4 GPU önerilir (birkaç dakika). CPU veya TPU çalışma zamanında da çalışır:
> betik TPU'yu kullanmaz, işlemcide çalışır (önceki çalıştırma ~30 dakika sürdü).
> Hangi cihazda eğitildiği doğruluğu anlamlı ölçüde değiştirmez; `sonuclar.json` içine kaydedilir.

## 0. Ayarlar

In [ ]:
# Proje deposu
REPO_URL = "https://github.com/rumeysakolip/bert-tiny-fpga.git"

# Drive'daki proje klasörü ve bu deneyin çıktı klasörü
DRIVE_ROOT = "/content/drive/MyDrive/bert-tiny-fpga"
OUT_DIR    = f"{DRIVE_ROOT}/deneyler/fp32_baz"

# True: Drive'daki eski sonuçların üzerine yazarak yeniden eğitir. False: sonuç varsa eğitimi atlar
YENIDEN_EGIT = True
# True: GPU yoksa durdurur. False: GPU yoksa uyarı verip işlemcide devam eder
GPU_ZORUNLU = False

## 1. Drive'ı bağla ve cihazı kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, torch
os.makedirs(OUT_DIR, exist_ok=True)
print("Çıktı klasörü:", OUT_DIR)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    msg = "GPU bulunamadı (TPU seçiliyse de eğitim işlemcide yapılır). Daha hızlı için: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU."
    if GPU_ZORUNLU:
        raise RuntimeError(msg)
    print("UYARI:", msg, "(işlemcide devam ediliyor, ~30 dakika sürebilir)")

## 2. Kodu indir ve ortamı kur

In [ ]:
%cd /content
if not os.path.exists("/content/bert-tiny-fpga"):
    !git clone {REPO_URL}
else:
    !git -C /content/bert-tiny-fpga pull
%cd /content/bert-tiny-fpga
!pip install -q -r requirements.txt
!git log -1 --oneline

In [ ]:
import transformers, datasets, platform
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets    :", datasets.__version__)

## 3. FP32 baz modelin eğitimi (G3)

`--out_dir` Drive'daki klasörü gösterdiği için her yeni en iyi epoch'ta `model/` Drive'a yazılır;
`sonuclar.json` ve `dogrulama_tahminleri.csv` de eğitim bitince doğrudan Drive'a düşer.
Ekran çıktısı ayrıca `egitim_log.txt` olarak kaydedilir.

In [ ]:
sonuc_yolu = f"{OUT_DIR}/sonuclar.json"
if os.path.exists(sonuc_yolu) and not YENIDEN_EGIT:
    print("Drive'da sonuç zaten var, eğitim atlandı:", sonuc_yolu)
else:
    !python -u yazilim/egit_fp32.py --out_dir "{OUT_DIR}" 2>&1 | tee "{OUT_DIR}/egitim_log.txt"

## 4. Sonuçları kontrol et

In [ ]:
import json
r = json.load(open(f"{OUT_DIR}/sonuclar.json", encoding="utf-8"))
print(f"FP32 doğrulama doğruluğu: %{100*r['dogruluk']:.2f}  (en iyi epoch {r['en_iyi_epoch']})")
print(f"Parametre: {r['parametre_sayisi']:,}  |  FP32 boyut: {r['fp32_boyut_MB']} MB  |  Eğitim süresi: {r['egitim_suresi_sn']} sn")
print("Cihaz:", r["ortam"]["cihaz"], "|", r["ortam"]["gpu"])
for h in r["gecmis"]:
    print(h)

print("\nDrive'daki dosyalar:")
for kok, _, dosyalar in os.walk(OUT_DIR):
    for d in dosyalar:
        p = os.path.join(kok, d)
        print(f"  {os.path.relpath(p, OUT_DIR):45s} {os.path.getsize(p)/1024:10.1f} KB")

## 5. Drive'a yazımı tamamla
Colab Drive'a yazmayı arka planda senkronlar; oturumu kapatmadan önce bu hücreyi çalıştırın.

In [ ]:
drive.flush_and_unmount()
print("Drive senkronlandı. Dosyalar: MyDrive/bert-tiny-fpga/deneyler/fp32_baz")